# Analyse du Data Drift - PROJET_8

Ce notebook constitue le livrable d'analyse du data drift de la mission. Il reprend la logique mise en place dans le projet :

1. chargement des événements de prédiction stockés par l'API ;
2. extraction des features TOP30 réellement consommées par le modèle ;
3. chargement de la référence construite depuis le dataset préparé du projet P6 ;
4. calcul des indicateurs opérationnels ;
5. comparaison des distributions avec Evidently ;
6. interprétation des résultats et points de vigilance.

Le notebook peut lire les événements depuis PostgreSQL local ou depuis le fichier JSONL généré par l'API.


## 1. Architecture analysée

```text
API FastAPI
  -> MonitoringService
  -> logs/api_predictions.jsonl
  -> import PostgreSQL local
  -> scripts/analyze_monitoring_logs.py
  -> Evidently DataDriftPreset
  -> reports/monitoring/data_drift_report.html
```

La référence de drift est générée par `scripts/build_monitoring_reference.py` à partir des données préparées du projet P6. Les données courantes viennent des appels API simulés ou réels.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd  # noqa: E402
from IPython.display import HTML, display  # noqa: E402

from scripts.analyze_monitoring_logs import (  # noqa: E402
    DEFAULT_DATABASE_URL,
    DEFAULT_LOG_PATH,
    DEFAULT_REFERENCE_PATH,
    extract_current_features,
    load_feature_names,
    load_jsonl,
    load_postgres_events,
    load_reference,
    run_drift_report,
    summarize_operational_metrics,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

OUTPUT_DIR = PROJECT_ROOT / "reports" / "monitoring" / "notebook"

print(f"Projet : {PROJECT_ROOT}")
print(f"R?f?rence : {DEFAULT_REFERENCE_PATH}")
print(f"Logs JSONL : {DEFAULT_LOG_PATH}")
print(f"Sorties notebook : {OUTPUT_DIR}")

Projet : C:\Users\kevin\Documents\PROJET_8
Référence : C:\Users\kevin\Documents\PROJET_8\monitoring\reference\top30_reference.parquet
Logs JSONL : C:\Users\kevin\Documents\PROJET_8\logs\api_predictions.jsonl
Sorties notebook : C:\Users\kevin\Documents\PROJET_8\reports\monitoring\notebook


## 2. Chargement des événements observés

Le notebook privilégie PostgreSQL, car c'est la solution de stockage structurée mise en place pour le PoC. Si la base locale n'est pas disponible, le fichier JSONL produit par l'API peut servir de source de secours.


In [2]:
source = "postgres"
try:
    events = load_postgres_events(DEFAULT_DATABASE_URL)
except Exception as error:
    print(f"PostgreSQL indisponible, bascule sur JSONL : {error}")
    source = "jsonl"
    events = load_jsonl(DEFAULT_LOG_PATH)

print(f"Source utilisée : {source}")
print(f"Nombre d'événements chargés : {len(events)}")

pd.DataFrame(events).head()

Source utilisée : postgres
Nombre d'événements chargés : 1000


,timestamp,request_id,endpoint,status,client_id,model_version,features,score,threshold,prediction,decision,latency_ms,preprocessing_latency_ms,inference_latency_ms,error_type,error_message
0,2026-05-15T12:26:52.846598+00:00,b585a3b5-0a7d-4335-b761-b59730ede170,/predict/batch,success,456017,lightgbm_top30_optimized,"{'amt_credit': 81504.0, 'days_birth': -9961.0,...",0.814758,0.5,1,high_risk,1.792,0.203,1.589,None,None
1,2026-05-15T12:26:52.843575+00:00,b585a3b5-0a7d-4335-b761-b59730ede170,/predict/batch,success,456016,lightgbm_top30_optimized,"{'amt_credit': 269550.0, 'days_birth': -13963....",0.102612,0.5,0,low_risk,1.529,0.080,1.449,None,None
2,2026-05-15T12:26:52.841585+00:00,b585a3b5-0a7d-4335-b761-b59730ede170,/predict/batch,success,455841,lightgbm_top30_optimized,"{'amt_credit': 993082.5, 'days_birth': -13032....",0.357267,0.5,0,low_risk,1.622,0.181,1.441,None,None
3,2026-05-15T12:26:52.838058+00:00,b585a3b5-0a7d-4335-b761-b59730ede170,/predict/batch,success,455204,lightgbm_top30_optimized,"{'amt_credit': 1515415.5, 'days_birth': -13132...",0.328944,0.5,0,low_risk,1.760,0.297,1.462,None,None
4,2026-05-15T12:26:52.836056+00:00,b585a3b5-0a7d-4335-b761-b59730ede170,/predict/batch,success,455026,lightgbm_top30_optimized,"{'amt_credit': 153000.0, 'days_birth': -12080....",0.188448,0.5,0,low_risk,1.629,0.115,1.514,None,None


## 3. Synthèse opérationnelle

Avant d'analyser le drift, on vérifie la qualité opérationnelle du flux : volume, erreurs, latences, scores et décisions. Ces métriques servent aussi à détecter des problèmes de production non liés directement au drift.


In [3]:
operational_summary = summarize_operational_metrics(events)

summary_rows = [
    ("total_events", operational_summary["total_events"]),
    ("success_count", operational_summary["success_count"]),
    ("error_count", operational_summary["error_count"]),
    ("error_rate", operational_summary["error_rate"]),
    ("latency_mean_ms", operational_summary["latency_ms"]["mean"]),
    ("latency_p95_ms", operational_summary["latency_ms"]["p95"]),
    ("score_mean", operational_summary["scores"]["mean"]),
    ("score_p95", operational_summary["scores"]["p95"]),
]

display(pd.DataFrame(summary_rows, columns=["indicateur", "valeur"]))
display(
    pd.DataFrame.from_dict(
        operational_summary["decisions"],
        orient="index",
        columns=["volume"],
    )
)

,indicateur,valeur
0,total_events,1000.000
1,success_count,1000.000
2,error_count,0.000
3,error_rate,0.000
4,latency_mean_ms,3.973
5,latency_p95_ms,2.713
6,score_mean,0.387
7,score_p95,0.796


,volume
high_risk,314
low_risk,686


## 4. Préparation des données de drift

La comparaison porte uniquement sur les features TOP30 utilisées par le modèle. Les événements en erreur ne sont pas utilisés pour le drift, car ils ne contiennent pas de features modèle exploitables.


In [4]:
feature_names = load_feature_names()
current_features = extract_current_features(events, feature_names)
reference = load_reference(DEFAULT_REFERENCE_PATH, feature_names)

print(f"Nombre de features attendues : {len(feature_names)}")
print(f"Lignes de référence : {len(reference)}")
print(f"Lignes observées : {len(current_features)}")

display(current_features.head())

Nombre de features attendues : 30


Lignes de référence : 10000
Lignes observées : 1000


,ext_sources_mean,credit_to_annuity_ratio,days_birth,ext_source_1,ext_source_2,payment_rate,credit_to_goods_ratio,amt_annuity,days_employed,approved_cnt_payment_mean,ext_source_3,days_employed_perc,prev_cnt_payment_mean,new_active_debt_ratio,annuity_to_income_ratio,instal_amt_payment_sum,new_late_payment_ratio,own_car_age,buro_amt_credit_max_overdue_mean,code_gender,amt_goods_price,days_id_publish,active_days_credit_max,active_days_credit_enddate_max,amt_credit,instal_days_entry_payment_max,new_bureau_debt_ratio,instal_payment_diff_mean,instal_dpd_mean,cc_cnt_drawings_atm_current_mean
0,0.163016,9.288205,-9961.0,0.175484,0.267573,0.107663,1.132000,8775.0,-368.0,6.500,0.045992,0.036944,6.400,0.928084,0.162500,307490.220,1.522727,NaN,1136.025,0.0,72000.0,-1352.0,-40.0,580.0,81504.0,-9.0,0.821940,505.840909,1.522727,0.076923
1,0.670205,18.925750,-13963.0,NaN,0.527587,0.052838,1.198000,14242.5,-4838.0,6.000,0.812823,0.346487,6.000,0.000000,0.105500,71024.490,0.000000,6.0,0.000,1.0,225000.0,-2880.0,-2750.0,-1644.0,269550.0,-325.0,0.000000,0.000000,0.000000,NaN
2,0.404673,25.132103,-13032.0,NaN,0.238429,0.039790,1.087118,39514.5,-2780.0,20.000,0.570917,0.213321,18.000,0.052138,0.195133,1846569.060,0.800000,8.0,29758.530,0.0,913500.0,-1955.0,-152.0,31067.0,993082.5,-49.0,0.009229,-7251.009000,0.800000,NaN
3,0.409858,34.202620,-13132.0,0.230352,0.651803,0.029238,1.118801,44307.0,-2574.0,8.625,0.347418,0.196010,8.625,0.546048,0.228977,1294275.915,0.052632,7.0,4236.750,1.0,1354500.0,-4779.0,-103.0,10040.0,1515415.5,-132.0,0.449700,232.507303,0.052632,NaN
4,0.531617,9.214092,-12080.0,0.328314,0.597481,0.108529,1.000000,16605.0,-351.0,10.000,0.669057,0.029056,10.000,0.976705,0.153750,266529.780,0.150000,NaN,0.000,0.0,153000.0,-4428.0,-84.0,1742.0,153000.0,-33.0,0.744801,0.000000,0.150000,NaN


## 5. Analyse Evidently

Le projet utilise `DataDriftPreset`. Evidently sélectionne automatiquement une méthode adaptée à chaque variable. Dans les rapports générés, les variables numériques sont principalement comparées avec la distance de Wasserstein normalisée ; la variable discrète `code_gender` est comparée avec Jensen-Shannon.


In [5]:
drift_summary = run_drift_report(reference, current_features, OUTPUT_DIR)

print(f"Statut : {drift_summary['status']}")
print(f"Drift détecté : {drift_summary.get('drift_detected')}")
print(f"Colonnes driftées : {drift_summary.get('drifted_columns_count')}")
print(f"Part de colonnes driftées : {drift_summary.get('drifted_columns_share')}")
print(f"Rapport HTML : {drift_summary.get('html_report_path')}")

Statut : computed
Drift détecté : True
Colonnes driftées : 1.0
Part de colonnes driftées : 0.03333333333333333
Rapport HTML : C:\Users\kevin\Documents\PROJET_8\reports\monitoring\notebook\data_drift_report.html


In [6]:
columns = pd.DataFrame(drift_summary.get("columns", []))
if not columns.empty:
    columns = columns.sort_values(["value", "column"], ascending=[False, True])
    display(columns.head(30))
else:
    print("Aucune métrique de colonne disponible.")

,column,method,threshold,value
29,cc_cnt_drawings_atm_current_mean,Wasserstein distance (normed),0.1,0.147439
22,active_days_credit_max,Wasserstein distance (normed),0.1,0.099937
17,own_car_age,Wasserstein distance (normed),0.1,0.094468
13,new_active_debt_ratio,Wasserstein distance (normed),0.1,0.088665
20,amt_goods_price,Wasserstein distance (normed),0.1,0.085630
24,amt_credit,Wasserstein distance (normed),0.1,0.082410
25,instal_days_entry_payment_max,Wasserstein distance (normed),0.1,0.080802
26,new_bureau_debt_ratio,Wasserstein distance (normed),0.1,0.074141
1,credit_to_annuity_ratio,Wasserstein distance (normed),0.1,0.071411
21,days_id_publish,Wasserstein distance (normed),0.1,0.067921


## 6. Lecture des résultats

Points d'interprétation :

- un drift global indique qu'au moins une partie des variables observées s'écarte de la référence ;
- une variable driftée n'est pas automatiquement une régression du modèle, mais un signal à investiguer ;
- l'analyse doit être relue avec le volume d'événements, la période d'observation et la population scorée ;
- les métriques de latence et d'erreur complètent le drift pour distinguer problème opérationnel et changement de distribution.

Dans le PoC local, l'objectif est de démontrer la chaîne complète : stockage des événements, comparaison à une référence et génération d'un rapport interprétable.


In [7]:
if drift_summary.get("html_report_path"):
    report_path = Path(drift_summary["html_report_path"])
    display(HTML(f'<p>Rapport Evidently généré : <code>{report_path}</code></p>'))

interpretation = {
    "source": source,
    "events_analyzed": len(events),
    "current_feature_rows": len(current_features),
    "reference_rows": len(reference),
    "drift_detected": drift_summary.get("drift_detected"),
    "drifted_columns_count": drift_summary.get("drifted_columns_count"),
    "drifted_columns_share": drift_summary.get("drifted_columns_share"),
    "error_rate": operational_summary.get("error_rate"),
    "latency_p95_ms": operational_summary["latency_ms"].get("p95"),
}

pd.DataFrame.from_dict(interpretation, orient="index", columns=["valeur"])

,valeur
source,postgres
events_analyzed,1000
current_feature_rows,1000
reference_rows,10000
drift_detected,True
drifted_columns_count,1.0
drifted_columns_share,0.033333
error_rate,0.0
latency_p95_ms,2.713


## 7. Conclusion

Le dispositif répond au besoin de monitoring ML de l'étape 3 :

- les appels API sont journalisés dans un format structuré ;
- les inputs modèle, outputs et temps d'exécution sont stockés ;
- PostgreSQL permet une analyse ultérieure des événements ;
- Evidently compare les données observées à une référence stable ;
- les rapports générés permettent d'identifier les variables à surveiller ;
- le dashboard local donne une lecture opérationnelle du système.

Pour une production réelle, les prochaines améliorations seraient l'ajout d'une politique de rétention, d'alertes automatiques et d'un stockage managé.
